In [1]:
import os, sys
sys.path.insert(0, "..")  # so `src` imports work

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType, TimestampType
)

spark = (
    SparkSession.builder
    .appName("01-explore")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

Spark 3.5.1


In [2]:
schema = StructType([
    StructField("VendorID", IntegerType()),
    StructField("tpep_pickup_datetime", TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count", IntegerType()),
    StructField("trip_distance", DoubleType()),
    StructField("pickup_longitude", DoubleType()),
    StructField("pickup_latitude", DoubleType()),
    StructField("RatecodeID", IntegerType()),
    StructField("store_and_fwd_flag", StringType()),
    StructField("dropoff_longitude", DoubleType()),
    StructField("dropoff_latitude", DoubleType()),
    StructField("payment_type", IntegerType()),
    StructField("fare_amount", DoubleType()),
    StructField("extra", DoubleType()),
    StructField("mta_tax", DoubleType()),
    StructField("tip_amount", DoubleType()),
    StructField("tolls_amount", DoubleType()),
    StructField("improvement_surcharge", DoubleType()),
    StructField("total_amount", DoubleType()),
])

df = (spark.read
      .option("header", True)
      .schema(schema)
      .csv("../data/yellow_tripdata_2016-03.csv"))

df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [3]:
print(f"Rows: {df.count():,}")
df.show(5, truncate=False)

Rows: 12,210,952
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |RatecodeID|store_and_fwd_flag|dropoff_longitude |dropoff_latitude |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|1       |2016-03-01 00:00:00 |2016-03-01 00:07:55  |1              |2.5          |-73.97674560546875|40.76515197753906 |1

In [4]:
df.select(
    "passenger_count", "trip_distance",
    "fare_amount", "tip_amount", "total_amount"
).summary("count", "min", "25%", "50%", "75%", "max").show()

+-------+---------------+-------------+-----------+----------+------------+
|summary|passenger_count|trip_distance|fare_amount|tip_amount|total_amount|
+-------+---------------+-------------+-----------+----------+------------+
|  count|       12210952|     12210952|   12210952|  12210952|    12210952|
|    min|              0|          0.0|     -376.0|     -20.0|      -376.3|
|    25%|              1|          1.0|        6.5|       0.0|        8.75|
|    50%|              1|          1.7|        9.5|      1.35|        11.8|
|    75%|              2|         3.18|       14.5|      2.36|       17.76|
|    max|              9| 1.90726288E7|  429496.72|     850.0|   429562.25|
+-------+---------------+-------------+-----------+----------+------------+



In [5]:
(df.write
   .mode("overwrite")
   .parquet("../data/yellow_2016_03.parquet"))
print("Parquet written.")

Parquet written.


In [6]:
spark.stop()